# 第02章：Program ID 与执行模型

## 前置知识
- 第01章：Hello Triton

## 学习目标
- 深入理解 `tl.program_id(axis)` 和多维 grid
- 掌握 Triton 的执行模型与 CUDA 的对应关系
- 学会用 1D 和 2D grid 处理不同维度的数据

In [1]:
import torch
import triton
import triton.language as tl

## 2.1 Program ID 详解

Triton 的 Grid 最多支持 **3个维度**（和 CUDA 一样）：

```
1D Grid:  grid = (num_programs_x,)
2D Grid:  grid = (num_programs_x, num_programs_y)
3D Grid:  grid = (num_programs_x, num_programs_y, num_programs_z)
```

对应的 program ID：
```
tl.program_id(0)  →  x 维度的索引  (类似 CUDA blockIdx.x)
tl.program_id(1)  →  y 维度的索引  (类似 CUDA blockIdx.y)
tl.program_id(2)  →  z 维度的索引  (类似 CUDA blockIdx.z)
```

### 1D Grid 示例

```
grid = (4,)  →  4 个 program 实例

Program:    [P0]  [P1]  [P2]  [P3]
pid(0):      0     1     2     3
```

### 2D Grid 示例

```
grid = (3, 2)  →  3×2 = 6 个 program 实例

           pid(0)=0   pid(0)=1   pid(0)=2
pid(1)=0  [P(0,0)]   [P(1,0)]   [P(2,0)]
pid(1)=1  [P(0,1)]   [P(1,1)]   [P(2,1)]
```

## 2.2 实例：用 2D Grid 处理矩阵

对于 2D 矩阵操作（如矩阵加法），用 2D grid 更自然：

```
矩阵 (M×N):
┌─────────┬─────────┬─────────┐
│ (0,0)   │ (0,1)   │ (0,2)   │  ← pid_y=0
│ BLOCK   │ BLOCK   │ BLOCK   │
├─────────┼─────────┼─────────┤
│ (1,0)   │ (1,1)   │ (1,2)   │  ← pid_y=1
│ BLOCK   │ BLOCK   │ BLOCK   │
└─────────┴─────────┴─────────┘
  pid_x=0   pid_x=1   pid_x=2

grid = (3, 2)  →  x维3个, y维2个
每个 program 处理一个 BLOCK_M × BLOCK_N 的子块
```

In [2]:
@triton.jit
def matrix_add_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N,           # 矩阵维度
    stride_m, stride_n,  # 内存步长
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    """
    2D 矩阵加法: C[m, n] = A[m, n] + B[m, n]
    使用 2D grid，每个 program 处理一个 BLOCK_M x BLOCK_N 的子块。
    """
    # 获取 2D program ID
    pid_m = tl.program_id(0)  # 行方向
    pid_n = tl.program_id(1)  # 列方向
    
    # 计算当前块的起始行/列
    # 例如: pid_m=1, BLOCK_M=32 → 处理第 32~63 行
    row_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    col_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    
    # 创建 2D mask
    row_mask = row_offsets < M
    col_mask = col_offsets < N
    
    # 计算 2D 内存偏移
    # offsets 是一个 BLOCK_M x BLOCK_N 的 2D 索引矩阵
    # row_offsets[:, None] 形状为 (BLOCK_M, 1)
    # col_offsets[None, :] 形状为 (1, BLOCK_N)
    # 广播后得到 (BLOCK_M, BLOCK_N)
    offsets = row_offsets[:, None] * stride_m + col_offsets[None, :] * stride_n
    mask = row_mask[:, None] & col_mask[None, :]
    
    # 加载、计算、存储
    a = tl.load(a_ptr + offsets, mask=mask, other=0.0)
    b = tl.load(b_ptr + offsets, mask=mask, other=0.0)
    c = a + b
    tl.store(c_ptr + offsets, c, mask=mask)

In [3]:
def matrix_add(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """2D 矩阵加法包装函数"""
    assert a.shape == b.shape
    M, N = a.shape
    c = torch.empty_like(a)
    
    BLOCK_M = 32
    BLOCK_N = 32
    
    # 2D grid: (M方向的块数, N方向的块数)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    
    matrix_add_kernel[grid](
        a, b, c,
        M, N,
        a.stride(0), a.stride(1),  # stride_m, stride_n
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
    )
    return c

# 测试
M, N = 127, 255  # 故意用非对齐的尺寸
a = torch.randn(M, N, device='cuda')
b = torch.randn(M, N, device='cuda')
c_triton = matrix_add(a, b)
c_torch = a + b

print(f"矩阵形状: ({M}, {N})")
print(f"结果正确: {torch.allclose(c_triton, c_torch)}")
print(f"最大误差: {(c_triton - c_torch).abs().max().item():.2e}")

矩阵形状: (127, 255)
结果正确: True
最大误差: 0.00e+00


## 2.3 stride 详解

**stride**（步长）描述了在内存中移动一个维度需要跳过多少元素。

```
矩阵 A (3×4), Row-Major 存储:

逻辑视图:          内存布局 (1D):
┌──┬──┬──┬──┐      [a00, a01, a02, a03, a10, a11, a12, a13, a20, a21, a22, a23]
│00│01│02│03│       ├───── row 0 ──────┤├───── row 1 ──────┤├───── row 2 ──────┤
├──┼──┼──┼──┤
│10│11│12│13│      stride_m = 4  (移到下一行要跳 4 个元素)
├──┼──┼──┼──┤      stride_n = 1  (移到下一列要跳 1 个元素)
│20│21│22│23│
└──┴──┴──┴──┘      A[i][j] 的内存地址 = base + i * stride_m + j * stride_n
```

PyTorch 中获取 stride：
```python
a = torch.randn(3, 4)
a.stride()  # (4, 1) → stride_m=4, stride_n=1
```

为什么需要 stride？因为矩阵在内存中是**一维连续存储**的，stride 帮助我们计算 2D 索引对应的 1D 内存地址。

## 2.4 `tl.num_programs` — 获取 Grid 大小

`tl.num_programs(axis)` 返回指定维度上的 program 总数，类似 CUDA 的 `gridDim.x`。

这在某些高级模式中很有用，比如循环处理超过 grid 大小的数据：

In [4]:
@triton.jit
def grid_stride_kernel(
    a_ptr, b_ptr, c_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    """
    Grid-Stride Loop 模式：
    当数据量 > grid * BLOCK_SIZE 时，每个 program 循环处理多个块。
    这类似 CUDA 的 grid-stride loop 模式。
    """
    pid = tl.program_id(0)
    num_programs = tl.num_programs(0)  # grid 大小
    
    # 每个 program 从 pid 开始，每次跳 num_programs 个块
    # 例如: grid=4, pid=1 → 处理块 1, 5, 9, 13, ...
    block_id = pid
    while block_id * BLOCK_SIZE < n_elements:
        offsets = block_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements
        a = tl.load(a_ptr + offsets, mask=mask)
        b = tl.load(b_ptr + offsets, mask=mask)
        tl.store(c_ptr + offsets, a + b, mask=mask)
        block_id += num_programs  # 跳到下一个要处理的块

# 用少量 program 处理大量数据
n = 100000
a = torch.randn(n, device='cuda')
b = torch.randn(n, device='cuda')
c = torch.empty_like(a)

# 只启动 32 个 program 处理 100000 个元素
grid_stride_kernel[(32,)](a, b, c, n, BLOCK_SIZE=1024)
print(f"Grid-stride loop 正确性: {torch.allclose(c, a + b)}")

Grid-stride loop 正确性: True


## 2.5 Triton 与 CUDA 执行模型对照表

```
                CUDA                    Triton
        ┌─────────────────┐     ┌─────────────────┐
        │     Grid        │     │     Grid        │
        │  ┌───┬───┬───┐  │     │  ┌───┬───┬───┐  │
        │  │Blk│Blk│Blk│  │     │  │ P │ P │ P │  │
        │  │ 0 │ 1 │ 2 │  │     │  │ 0 │ 1 │ 2 │  │
        │  └───┴───┴───┘  │     │  └───┴───┴───┘  │
        │                 │     │                 │
        │  每个 Block:     │     │  每个 Program:   │
        │  ┌─┬─┬─┬─┬─┬─┐ │     │  处理一整块数据  │
        │  │T│T│T│T│T│T│ │     │  (编译器管理线程) │
        │  └─┴─┴─┴─┴─┴─┘ │     │                 │
        │  你管理每个线程  │     │  你只写块级逻辑  │
        └─────────────────┘     └─────────────────┘
```

| CUDA 概念 | Triton 对应 | 说明 |
|-----------|------------|------|
| `gridDim` | `tl.num_programs(axis)` | Grid 大小 |
| `blockIdx` | `tl.program_id(axis)` | 当前块/程序的索引 |
| `blockDim` | 编译器管理 | 你不需要关心 |
| `threadIdx` | 编译器管理 | 你不需要关心 |
| 1 thread → 1 element | 1 program → BLOCK elements | 编程粒度的核心区别 |

## 2.6 实用技巧：用 lambda grid 实现动态 grid

当 grid 大小依赖于编译时常量（如 BLOCK_SIZE），可以用 lambda 函数：

In [5]:
@triton.jit
def add_kernel_v2(a_ptr, b_ptr, c_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    tl.store(c_ptr + offsets, a + b, mask=mask)

n = 1000
a = torch.randn(n, device='cuda')
b = torch.randn(n, device='cuda')
c = torch.empty_like(a)

# 使用 lambda grid —— grid 大小会根据 meta['BLOCK_SIZE'] 自动计算
# meta 字典包含所有 tl.constexpr 参数
grid = lambda meta: (triton.cdiv(n, meta['BLOCK_SIZE']),)
add_kernel_v2[grid](a, b, c, n, BLOCK_SIZE=256)

print(f"Lambda grid 正确性: {torch.allclose(c, a + b)}")

Lambda grid 正确性: True


## 总结

| 概念 | API | 说明 |
|------|-----|------|
| Program ID | `tl.program_id(axis)` | 0, 1, 2 三个轴 |
| Grid 大小 | `tl.num_programs(axis)` | 各轴的 program 数 |
| 2D 索引 | `row[:, None] * stride_m + col[None, :]` | 广播构建 2D 偏移 |
| Stride | `tensor.stride(dim)` | 维度步长 |
| Lambda Grid | `lambda meta: (cdiv(n, meta['BS']),)` | 动态 grid |

## 练习

1. **3D Grid**：实现一个 3D 张量加法（batch × M × N），使用 3D grid
2. **转置**：用 2D grid 实现矩阵转置 B = A^T
3. **对角线提取**：用 1D grid 提取方阵的对角线元素